# Unified Amputee MAT Exercise Classifier

This notebook uses only the original Amputee MAT recordings. Excel exports are not loaded.

The available 22-sensor CyberGlove signal supports:

- Exercise B: 17 hand and wrist actions
- Exercise C: 23 grasps and functional actions
- Rest

Exercise D has 9 actions but its MAT files contain no `glove` variable; D must be modeled separately from EMG/force. The current model therefore learns Amputee MAT glove-frequency patterns for B/C and predicts the named exercise/subexercise label. This is a kinematic-sensor experiment, not EMG-only prosthetic control.

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import classification_report

# Repo root is the parent of notebooks/ when running from there, else cwd itself.
_root = Path.cwd() if (Path.cwd() / 'utils.py').exists() else Path.cwd().parent
sys.path.insert(0, str(_root))
from utils import (
    ACTIONS, set_seed, find_repo_root, get_data_root,
    load_mat_recording, make_frequency_windows,
    subject_grouped_holdout, cap_and_shuffle_classes, exercise_labels_from_classes,
    fit_frequency_model, predict_hierarchical, classification_summary,
)

SEED = 42
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = find_repo_root()
DATA_ROOT = get_data_root(ROOT)
WINDOW_SAMPLES = 400
STRIDE_SAMPLES = 100
FFT_BINS = 32
CHANNELS = 22
MIN_PURITY = 0.80
print('device:', DEVICE, 'classes:', 1 + sum(len(v) for v in ACTIONS.values()))


device: cpu classes: 62


## MAT-only file-to-taxonomy mapping and sensor availability

This canonical classifier uses only the original Amputee MAT recordings. Excel exports are intentionally excluded from the analysis.

- Amputee E1 -> Exercise B: 17 actions plus Rest
- Amputee E2 -> Exercise C: 23 actions plus Rest; raw labels 18-40 are remapped to local labels 1-23
- Amputee E3 -> Exercise D: 9 actions, but these files contain no `glove` variable, so they cannot enter this 22-sensor frequency experiment
- Label 0 -> Rest

The current 22-sensor run therefore contains Exercise B and C plus Rest. Exercise D requires a separate EMG/force experiment because its MAT files do not contain the 22 glove channels.

In [2]:
paths = sorted((DATA_ROOT / 'Amputee').glob('*_E*_A1.mat'))
records = []
for path in paths:
    try:
        recording = load_mat_recording(path)
        records.append(make_frequency_windows(
            recording, path, cohort='Amputee', channels=CHANNELS,
            window_samples=WINDOW_SAMPLES, stride_samples=STRIDE_SAMPLES,
            fft_bins=FFT_BINS, min_purity=MIN_PURITY,
        ))
    except Exception as error:
        print('skipped', path.name, error)
X = np.concatenate([item[0] for item in records])
target_names = np.concatenate([item[1] for item in records])
groups = np.concatenate([item[2] for item in records])
names = np.concatenate([item[3] for item in records])
classes = sorted(np.unique(target_names))
class_to_id = {name: index for index, name in enumerate(classes)}
y = np.asarray([class_to_id[name] for name in target_names])
print('Amputee MAT-only glove windows:', X.shape, 'classes:', len(classes), 'exercises:', sorted({name.split(':')[0] for name in classes}))


skipped S10_E3_A1.mat S10_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S11_E3_A1.mat S11_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S1_E3_A1.mat S1_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S2_E3_A1.mat S2_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S3_E3_A1.mat S3_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S4_E3_A1.mat S4_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S5_E3_A1.mat S5_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S6_E3_A1.mat S6_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S7_E3_A1.mat S7_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S8_E3_A1.mat S8_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S9_E3_A1.mat S9_E3_A1.mat has no 22-sensor glove input; missing ['glove']
Amputee MAT-only glove windows: (443849, 704) classes: 42 exercises: ['B', 'C']


In [3]:
subjects = np.asarray([group.split(':', 1)[0] for group in groups])
train, test, train_groups, test_groups = subject_grouped_holdout(groups, subjects, SEED, test_fraction=0.2)
selected = cap_and_shuffle_classes(y, train, SEED, max_per_class=1000)
X_train, y_train = X[selected], y[selected]
X_test, y_test = X[test], y[test]
exercise_ids, exercise_to_id, y_exercise = exercise_labels_from_classes(classes, y)

exercise_model = fit_frequency_model(X_train, y_exercise[selected], len(exercise_ids), DEVICE)
action_models = {}
for exercise in exercise_ids:
    global_ids = [index for index, name in enumerate(classes) if name.startswith(f'{exercise}:')]
    local_map = {global_id: local for local, global_id in enumerate(global_ids)}
    keep = np.asarray([classes[label].startswith(f'{exercise}:') for label in y_train])
    local_labels = np.asarray([local_map[label] for label in y_train[keep]])
    action_models[exercise] = (fit_frequency_model(X_train[keep], local_labels, len(global_ids), DEVICE), global_ids)


model epochs 10 classes 2
model epochs 20 classes 2
model epochs 30 classes 2
model epochs 40 classes 2
model epochs 50 classes 2
model epochs 60 classes 2
model epochs 70 classes 2
model epochs 80 classes 2
model epochs 90 classes 2
model epochs 100 classes 2
model epochs 110 classes 2
model epochs 120 classes 2
model epochs 130 classes 2
model epochs 140 classes 2
model epochs 150 classes 2
model epochs 160 classes 2
model epochs 170 classes 2
model epochs 180 classes 2
model epochs 190 classes 2
model epochs 200 classes 2
model epochs 10 classes 18
model epochs 20 classes 18
model epochs 30 classes 18
model epochs 40 classes 18
model epochs 50 classes 18
model epochs 60 classes 18
model epochs 70 classes 18
model epochs 80 classes 18
model epochs 90 classes 18
model epochs 100 classes 18
model epochs 110 classes 18
model epochs 120 classes 18
model epochs 130 classes 18
model epochs 140 classes 18
model epochs 150 classes 18
model epochs 160 classes 18
model epochs 170 classes 18
mo

In [4]:
true_exercises = y_exercise[test]
predicted_exercises, predictions = predict_hierarchical(exercise_model, action_models, exercise_ids, X_test, DEVICE)
metrics = {
    **classification_summary(y_test, predictions),
    'exercise_accuracy': float((predicted_exercises == true_exercises).mean()),
    'classes': len(classes),
    'test_windows': len(y_test),
}
print(metrics)
print(classification_report(y_test, predictions, target_names=classes, zero_division=0))


{'accuracy': 0.09334834516815856, 'balanced_accuracy': 0.10061451118164674, 'macro_f1': 0.08389803616976364, 'exercise_accuracy': 0.8164580244574002, 'classes': 42, 'test_windows': 89707}
              precision    recall  f1-score   support

         B:0       0.52      0.18      0.27     12413
         B:1       0.12      0.08      0.10      1446
        B:10       0.19      0.11      0.14      1434
        B:11       0.08      0.02      0.04      1470
        B:12       0.02      0.03      0.02      1397
        B:13       0.14      0.05      0.07      1466
        B:14       0.21      0.07      0.11      1383
        B:15       0.14      0.04      0.07      1542
        B:16       0.19      0.13      0.16      1498
        B:17       0.04      0.26      0.06      1315
         B:2       0.13      0.14      0.14      1469
         B:3       0.16      0.10      0.12      1324
         B:4       0.10      0.07      0.09      1422
         B:5       0.03      0.15      0.06      1388
 

## Interpretation

This notebook predicts the named exercise and subexercise from frequency patterns in the 22 glove sensors. Because the glove directly measures hand kinematics, this is not evidence that EMG alone can control a prosthesis. The exercise-specific label mapping is explicit, and the split is by complete subject-repetition groups to prevent overlapping-window leakage.

## Complete 49-movement EMG classifier

The glove-frequency model above covers B/C only because Exercise D has no glove variable. The complete Amputee action task uses the 12 EMG channels and includes B, C, D, plus Rest. Raw label normalization is:

```text
E1: 0..17  -> B: Rest + 17 actions
E2: 0,18..40 -> C: Rest + 23 actions, mapped to local 0..23
E3: 0,41..49 -> D: Rest + 9 actions, mapped to local 0..9
```

This is the model to use when the input is an EMG signal and the requested output is any of the 49 movements. The 22-channel glove model should not be used for D.
